# Improve NEM Spot Price Prediction Modelling

This notebook is designed for the Phase 2 training workflow using the prepared `train.csv` and `val.csv` files.

It focuses on:
- validating against a strong persistence baseline
- comparing small same-region models before larger models
- using MAE and RMSE only
- saving models only if they beat the baseline

The target columns are:
- `NSW1_target_price_t_plus_6`
- `QLD1_target_price_t_plus_6`
- `SA1_target_price_t_plus_6`
- `TAS1_target_price_t_plus_6`
- `VIC1_target_price_t_plus_6`

In [ ]:
# Run this once in Colab if the packages are not already installed.
!pip install -q lightgbm scikit-learn joblib matplotlib seaborn

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lightgbm import LGBMRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

In [ ]:
# Update these paths if your files live elsewhere in Colab or Google Drive.
TRAIN_PATH = "train.csv"
VAL_PATH = "val.csv"
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

regions = ["NSW1", "QLD1", "SA1", "TAS1", "VIC1"]
target_cols = [f"{region}_target_price_t_plus_6" for region in regions]
current_price_cols = {target: target.replace("_target_price_t_plus_6", "_price_dollar_per_mwh") for target in target_cols}

print(train_df.shape, val_df.shape)
train_df.head(2)

In [ ]:
def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)


def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5


def get_same_region_small_features(region: str) -> list[str]:
    return [
        f"{region}_price_dollar_per_mwh",
        f"{region}_demand_mw",
        f"{region}_renewables_pct",
        f"{region}_price_lag_1",
        f"{region}_price_lag_2",
        f"{region}_price_lag_6",
        f"{region}_price_lag_12",
        f"{region}_demand_rollmean_6",
        f"{region}_demand_rollmean_12",
        f"{region}_wind_rollmean_6",
        f"{region}_wind_rollmean_12",
        "hour",
        "day_of_week",
        "month",
        "is_weekend",
        "is_business_hour",
    ]


def get_full_feature_cols(df: pd.DataFrame) -> list[str]:
    return [col for col in df.columns if col not in {"timestamp", *target_cols}]


def evaluate_predictions(y_true, y_pred) -> dict:
    return {
        "MAE": mae(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
    }


def model_summary_row(region: str, model_name: str, y_true, y_pred) -> dict:
    metrics = evaluate_predictions(y_true, y_pred)
    return {
        "region": region,
        "model": model_name,
        **metrics,
    }

## 1. Baseline And Target Diagnostics

The persistence baseline predicts:

`future price in 30 minutes = current price now`

Any ML model should beat this baseline before it is considered for saving.

In [ ]:
baseline_rows = []
distribution_rows = []

for region in regions:
    target = f"{region}_target_price_t_plus_6"
    current = f"{region}_price_dollar_per_mwh"

    y_true = val_df[target]
    y_pred = val_df[current]

    baseline_rows.append({
        "region": region,
        "target": target,
        "baseline_feature": current,
        **evaluate_predictions(y_true, y_pred),
    })

    s = val_df[target].dropna()
    distribution_rows.append({
        "region": region,
        "count": len(s),
        "min": s.min(),
        "p1": s.quantile(0.01),
        "p5": s.quantile(0.05),
        "median": s.median(),
        "p95": s.quantile(0.95),
        "p99": s.quantile(0.99),
        "max": s.max(),
        "mean": s.mean(),
        "std": s.std(),
        "negative_count": int((s < 0).sum()),
        "gt_300": int((s > 300).sum()),
        "gt_1000": int((s > 1000).sum()),
    })

baseline_df = pd.DataFrame(baseline_rows).sort_values("region").reset_index(drop=True)
dist_df = pd.DataFrame(distribution_rows).sort_values("region").reset_index(drop=True)

print("Naive baseline")
display(baseline_df)

print("Validation target distribution")
display(dist_df)

## 2. Sanity-Check Ladder

This section deliberately starts simple:
- baseline
- ridge with current price only
- ridge with small same-region features
- LightGBM with small same-region features
- optional LightGBM with all features

If a more complex model cannot beat the baseline, it should not be used in production.

In [ ]:
all_feature_cols = get_full_feature_cols(train_df)
results = []
predictions = {}
trained_models = {}

for region in regions:
    target = f"{region}_target_price_t_plus_6"
    current = f"{region}_price_dollar_per_mwh"
    small_features = get_same_region_small_features(region)

    y_train = train_df[target]
    y_val = val_df[target]

    # Baseline
    baseline_pred = val_df[current]
    results.append(model_summary_row(region, "baseline_current_price", y_val, baseline_pred))
    predictions[(region, "baseline_current_price")] = baseline_pred

    # Ridge: current price only
    ridge_current = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", Ridge(alpha=1.0)),
    ])
    ridge_current.fit(train_df[[current]], y_train)
    ridge_current_pred = ridge_current.predict(val_df[[current]])
    results.append(model_summary_row(region, "ridge_current_only", y_val, ridge_current_pred))
    predictions[(region, "ridge_current_only")] = ridge_current_pred
    trained_models[(region, "ridge_current_only")] = ridge_current

    # Ridge: small same-region feature set
    ridge_small = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", Ridge(alpha=10.0)),
    ])
    ridge_small.fit(train_df[small_features], y_train)
    ridge_small_pred = ridge_small.predict(val_df[small_features])
    results.append(model_summary_row(region, "ridge_small_features", y_val, ridge_small_pred))
    predictions[(region, "ridge_small_features")] = ridge_small_pred
    trained_models[(region, "ridge_small_features")] = ridge_small

    # LightGBM: small same-region feature set
    lgb_small = LGBMRegressor(
        objective="regression",
        n_estimators=300,
        learning_rate=0.03,
        num_leaves=15,
        min_child_samples=100,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        force_col_wise=True,
    )
    lgb_small.fit(train_df[small_features], y_train)
    lgb_small_pred = lgb_small.predict(val_df[small_features])
    results.append(model_summary_row(region, "lightgbm_small_features", y_val, lgb_small_pred))
    predictions[(region, "lightgbm_small_features")] = lgb_small_pred
    trained_models[(region, "lightgbm_small_features")] = lgb_small

    # LightGBM: all features
    lgb_full = LGBMRegressor(
        objective="regression",
        n_estimators=400,
        learning_rate=0.03,
        num_leaves=15,
        min_child_samples=150,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        force_col_wise=True,
    )
    lgb_full.fit(train_df[all_feature_cols], y_train)
    lgb_full_pred = lgb_full.predict(val_df[all_feature_cols])
    results.append(model_summary_row(region, "lightgbm_full_features", y_val, lgb_full_pred))
    predictions[(region, "lightgbm_full_features")] = lgb_full_pred
    trained_models[(region, "lightgbm_full_features")] = lgb_full

results_df = pd.DataFrame(results).sort_values(["region", "RMSE", "MAE"]).reset_index(drop=True)
display(results_df)

In [ ]:
comparison_df = results_df.merge(
    baseline_df[["region", "MAE", "RMSE"]].rename(columns={"MAE": "baseline_MAE", "RMSE": "baseline_RMSE"}),
    on="region",
    how="left",
)
comparison_df["MAE_improvement_vs_baseline"] = comparison_df["baseline_MAE"] - comparison_df["MAE"]
comparison_df["RMSE_improvement_vs_baseline"] = comparison_df["baseline_RMSE"] - comparison_df["RMSE"]

display(comparison_df.sort_values(["region", "RMSE_improvement_vs_baseline"], ascending=[True, False]))

In [ ]:
best_by_region = (
    comparison_df.sort_values(["region", "RMSE", "MAE"])
    .groupby("region", as_index=False)
    .first()
    .sort_values("region")
)

display(best_by_region)

fig, axes = plt.subplots(len(regions), 1, figsize=(14, 3 * len(regions)), sharex=False)

for ax, region in zip(axes, regions):
    subset = comparison_df[comparison_df["region"] == region].sort_values("RMSE")
    sns.barplot(data=subset, x="RMSE", y="model", ax=ax, palette="viridis")
    ax.set_title(f"{region} model comparison")
    ax.set_xlabel("Validation RMSE")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()

## 3. Visual Diagnostics

These charts help confirm whether the chosen model is actually tracking the short-horizon price path or just smoothing it too much.

In [ ]:
rows = []
for region in regions:
    chosen_model_name = best_by_region.loc[best_by_region["region"] == region, "model"].iloc[0]
    target = f"{region}_target_price_t_plus_6"
    sample = pd.DataFrame({
        "actual": val_df[target].head(300).values,
        "prediction": np.asarray(predictions[(region, chosen_model_name)])[:300],
        "baseline": val_df[f"{region}_price_dollar_per_mwh"].head(300).values,
    })
    sample["region"] = region
    sample["best_model"] = chosen_model_name
    rows.append(sample)

plot_df = pd.concat(rows, ignore_index=True)

fig, axes = plt.subplots(len(regions), 1, figsize=(16, 3 * len(regions)), sharex=False)
for ax, region in zip(axes, regions):
    region_df = plot_df[plot_df["region"] == region].reset_index(drop=True)
    ax.plot(region_df.index, region_df["actual"], label="actual", linewidth=1.8)
    ax.plot(region_df.index, region_df["prediction"], label=f"best model: {region_df['best_model'].iloc[0]}", linewidth=1.4)
    ax.plot(region_df.index, region_df["baseline"], label="baseline current price", linewidth=1.2, alpha=0.9)
    ax.set_title(region)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
feature_importance_frames = []

for region in regions:
    model_name = best_by_region.loc[best_by_region["region"] == region, "model"].iloc[0]
    model = trained_models.get((region, model_name))
    if model is None or not hasattr(model, "feature_importances_"):
        continue

    if model_name == "lightgbm_full_features":
        features = all_feature_cols
    else:
        features = get_same_region_small_features(region)

    fi = pd.DataFrame({
        "region": region,
        "feature": features,
        "importance": model.feature_importances_,
        "model": model_name,
    }).sort_values("importance", ascending=False)

    feature_importance_frames.append(fi.head(15))

if feature_importance_frames:
    feature_importance_df = pd.concat(feature_importance_frames, ignore_index=True)
    display(feature_importance_df)
else:
    print("No LightGBM model was the top model for any region, so there are no tree-based feature importances to display.")

## 4. Save Final Models

This cell only saves a region model if it beats the persistence baseline on validation RMSE.

If no trained model beats the baseline for a region, the notebook records that the baseline should be used instead.

In [ ]:
artifacts = []
final_models = {}

for region in regions:
    baseline_row = comparison_df[(comparison_df["region"] == region) & (comparison_df["model"] == "baseline_current_price")].iloc[0]
    candidate_rows = comparison_df[(comparison_df["region"] == region) & (comparison_df["model"] != "baseline_current_price")].sort_values(["RMSE", "MAE"])
    best_candidate = candidate_rows.iloc[0]

    if best_candidate["RMSE"] < baseline_row["RMSE"]:
        model_name = best_candidate["model"]
        model = trained_models[(region, model_name)]
        features = all_feature_cols if model_name == "lightgbm_full_features" else get_same_region_small_features(region)
        model_path = MODEL_DIR / f"{region.lower()}_{model_name}.pkl"
        metadata_path = MODEL_DIR / f"{region.lower()}_{model_name}_metadata.json"

        joblib.dump(model, model_path)
        metadata = {
            "region": region,
            "model_name": model_name,
            "target_column": f"{region}_target_price_t_plus_6",
            "feature_columns": features,
            "validation_mae": float(best_candidate["MAE"]),
            "validation_rmse": float(best_candidate["RMSE"]),
            "baseline_rmse": float(baseline_row["RMSE"]),
        }
        metadata_path.write_text(json.dumps(metadata, indent=2))

        final_models[region] = {
            "type": "trained_model",
            "model_name": model_name,
            "model_path": str(model_path),
            "metadata_path": str(metadata_path),
        }
        artifacts.append({
            "region": region,
            "final_choice": model_name,
            "saved_model": str(model_path),
            "saved_metadata": str(metadata_path),
        })
    else:
        final_models[region] = {
            "type": "baseline",
            "rule": f"predict {region}_target_price_t_plus_6 using current {region}_price_dollar_per_mwh",
        }
        artifacts.append({
            "region": region,
            "final_choice": "baseline_current_price",
            "saved_model": None,
            "saved_metadata": None,
        })

artifacts_df = pd.DataFrame(artifacts).sort_values("region").reset_index(drop=True)
display(artifacts_df)
final_models

## Notes

If the baseline still wins for most regions, that is a meaningful result. For a 30-minute horizon, persistence can be very strong.

In that case, the next experiments should be:
- more focused regional feature engineering
- regime or spike classification for volatile regions like `SA1`
- only then, adding more historical data

## 5. Residual And Blended Models For All States

This section trains a model for every region using a residual target:

`residual = future_price_30m - current_price`

The final prediction is blended with persistence:

`final_prediction = current_price + alpha * predicted_residual`

`alpha` is tuned on the validation set for each region. If the best `alpha` is near `0`, the model behaves almost like persistence while still preserving a model artifact for that region.

In [ ]:
from itertools import combinations


def add_enhanced_short_horizon_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

    for region in regions:
        current_price = f"{region}_price_dollar_per_mwh"
        lag_1 = f"{region}_price_lag_1"
        lag_2 = f"{region}_price_lag_2"
        lag_6 = f"{region}_price_lag_6"
        lag_12 = f"{region}_price_lag_12"
        demand = f"{region}_demand_mw"
        demand_roll_6 = f"{region}_demand_rollmean_6"
        demand_roll_12 = f"{region}_demand_rollmean_12"
        wind = f"{region}_gen_wind_mw"
        wind_roll_6 = f"{region}_wind_rollmean_6"
        wind_roll_12 = f"{region}_wind_rollmean_12"
        renewables = f"{region}_renewables_pct"

        df[f"{region}_price_ramp_1"] = df[current_price] - df[lag_1]
        df[f"{region}_price_ramp_2"] = df[current_price] - df[lag_2]
        df[f"{region}_price_ramp_6"] = df[current_price] - df[lag_6]
        df[f"{region}_price_ramp_12"] = df[current_price] - df[lag_12]
        df[f"{region}_price_ramp_1_to_6"] = df[lag_1] - df[lag_6]
        df[f"{region}_price_volatility"] = df[[current_price, lag_1, lag_2, lag_6, lag_12]].std(axis=1)

        df[f"{region}_demand_ramp_6"] = df[demand] - df[demand_roll_6]
        df[f"{region}_demand_ramp_12"] = df[demand] - df[demand_roll_12]
        df[f"{region}_wind_ramp_6"] = df[wind] - df[wind_roll_6]
        df[f"{region}_wind_ramp_12"] = df[wind] - df[wind_roll_12]
        df[f"{region}_wind_share_change"] = df[wind] * df[renewables] / 100.0

    for left, right in combinations(regions, 2):
        left_price = f"{left}_price_dollar_per_mwh"
        right_price = f"{right}_price_dollar_per_mwh"
        df[f"spread_{left}_{right}"] = df[left_price] - df[right_price]
        df[f"abs_spread_{left}_{right}"] = (df[left_price] - df[right_price]).abs()

    return df


def get_residual_feature_cols(region: str, df: pd.DataFrame) -> list[str]:
    base = get_same_region_small_features(region)
    same_region_extra = [
        f"{region}_price_ramp_1",
        f"{region}_price_ramp_2",
        f"{region}_price_ramp_6",
        f"{region}_price_ramp_12",
        f"{region}_price_ramp_1_to_6",
        f"{region}_price_volatility",
        f"{region}_demand_ramp_6",
        f"{region}_demand_ramp_12",
        f"{region}_wind_ramp_6",
        f"{region}_wind_ramp_12",
        f"{region}_wind_share_change",
        "hour_sin",
        "hour_cos",
        "dow_sin",
        "dow_cos",
    ]

    cross_region = []
    for other in regions:
        if other == region:
            continue
        cross_region.extend([
            f"{other}_price_dollar_per_mwh",
            f"{other}_price_lag_1",
            f"{other}_price_lag_6",
            f"{other}_demand_mw",
            f"{other}_renewables_pct",
        ])
        spread_name = f"spread_{region}_{other}"
        reverse_spread_name = f"spread_{other}_{region}"
        abs_spread_name = f"abs_spread_{region}_{other}"
        reverse_abs_spread_name = f"abs_spread_{other}_{region}"
        for name in [spread_name, reverse_spread_name, abs_spread_name, reverse_abs_spread_name]:
            if name in df.columns:
                cross_region.append(name)

    ordered = []
    seen = set()
    for col in [*base, *same_region_extra, *cross_region]:
        if col in df.columns and col not in seen and col not in target_cols and col != "timestamp":
            ordered.append(col)
            seen.add(col)

    return ordered


train_enhanced_df = add_enhanced_short_horizon_features(train_df)
val_enhanced_df = add_enhanced_short_horizon_features(val_df)

train_enhanced_df.shape, val_enhanced_df.shape

In [ ]:
alpha_grid = np.linspace(0.0, 1.0, 21)
residual_results = []
residual_predictions = {}
residual_models = {}
residual_feature_map = {}

for region in regions:
    target = f"{region}_target_price_t_plus_6"
    current = f"{region}_price_dollar_per_mwh"
    feature_cols = get_residual_feature_cols(region, train_enhanced_df)
    residual_feature_map[region] = feature_cols

    X_train = train_enhanced_df[feature_cols]
    X_val = val_enhanced_df[feature_cols]
    y_train_residual = train_enhanced_df[target] - train_enhanced_df[current]
    y_val_true = val_enhanced_df[target]
    baseline_pred = val_enhanced_df[current].to_numpy()

    model = LGBMRegressor(
        objective="regression_l1",
        n_estimators=600,
        learning_rate=0.02,
        num_leaves=7,
        min_child_samples=200,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        force_col_wise=True,
    )
    model.fit(X_train, y_train_residual)

    predicted_residual = model.predict(X_val)
    residual_predictions[(region, "residual_raw")] = predicted_residual
    residual_models[region] = model

    best_choice = None
    for alpha in alpha_grid:
        blended_pred = baseline_pred + alpha * predicted_residual
        row = {
            "region": region,
            "model": "lightgbm_residual_blend",
            "alpha": float(alpha),
            "feature_count": len(feature_cols),
            **evaluate_predictions(y_val_true, blended_pred),
        }
        if best_choice is None or row["RMSE"] < best_choice["RMSE"] or (
            np.isclose(row["RMSE"], best_choice["RMSE"]) and row["MAE"] < best_choice["MAE"]
        ):
            best_choice = row
            residual_predictions[(region, "residual_blend_best")] = blended_pred

    residual_results.append(best_choice)

residual_results_df = pd.DataFrame(residual_results).sort_values(["region", "RMSE", "MAE"]).reset_index(drop=True)
display(residual_results_df)

In [ ]:
residual_comparison_df = residual_results_df.merge(
    baseline_df[["region", "MAE", "RMSE"]].rename(columns={"MAE": "baseline_MAE", "RMSE": "baseline_RMSE"}),
    on="region",
    how="left",
)
residual_comparison_df["MAE_improvement_vs_baseline"] = residual_comparison_df["baseline_MAE"] - residual_comparison_df["MAE"]
residual_comparison_df["RMSE_improvement_vs_baseline"] = residual_comparison_df["baseline_RMSE"] - residual_comparison_df["RMSE"]

display(residual_comparison_df.sort_values(["RMSE_improvement_vs_baseline", "MAE_improvement_vs_baseline"], ascending=False))

fig, axes = plt.subplots(len(regions), 1, figsize=(16, 3 * len(regions)), sharex=False)
for ax, region in zip(axes, regions):
    sample = pd.DataFrame({
        "actual": val_enhanced_df[f"{region}_target_price_t_plus_6"].head(300).values,
        "baseline": val_enhanced_df[f"{region}_price_dollar_per_mwh"].head(300).values,
        "residual_blend": np.asarray(residual_predictions[(region, "residual_blend_best")])[:300],
    })
    alpha = residual_results_df.loc[residual_results_df["region"] == region, "alpha"].iloc[0]
    ax.plot(sample.index, sample["actual"], label="actual", linewidth=1.8)
    ax.plot(sample.index, sample["baseline"], label="baseline current price", linewidth=1.2)
    ax.plot(sample.index, sample["residual_blend"], label=f"residual blend alpha={alpha:.2f}", linewidth=1.4)
    ax.set_title(f"{region} residual blend diagnostics")
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

## 6. Save Residual-Blend Models For All States

This cell always writes a model artifact for every region.

If `alpha` is very small, the saved model behaves almost like the persistence baseline, but you still get one consistent inference contract across all five regions.

In [ ]:
residual_artifacts = []
residual_final_models = {}

for region in regions:
    model = residual_models[region]
    best_row = residual_results_df[residual_results_df["region"] == region].iloc[0]
    model_name = "lightgbm_residual_blend"
    model_path = MODEL_DIR / f"{region.lower()}_{model_name}.pkl"
    metadata_path = MODEL_DIR / f"{region.lower()}_{model_name}_metadata.json"

    joblib.dump(model, model_path)
    metadata = {
        "region": region,
        "model_name": model_name,
        "target_column": f"{region}_target_price_t_plus_6",
        "current_price_column": f"{region}_price_dollar_per_mwh",
        "feature_columns": residual_feature_map[region],
        "prediction_rule": "current_price + alpha * model_predicted_residual",
        "alpha": float(best_row["alpha"]),
        "validation_mae": float(best_row["MAE"]),
        "validation_rmse": float(best_row["RMSE"]),
        "baseline_mae": float(best_row["baseline_MAE"] if "baseline_MAE" in best_row else baseline_df.loc[baseline_df["region"] == region, "MAE"].iloc[0]),
        "baseline_rmse": float(best_row["baseline_RMSE"] if "baseline_RMSE" in best_row else baseline_df.loc[baseline_df["region"] == region, "RMSE"].iloc[0]),
    }
    metadata_path.write_text(json.dumps(metadata, indent=2))

    residual_final_models[region] = {
        "type": "trained_model",
        "model_name": model_name,
        "model_path": str(model_path),
        "metadata_path": str(metadata_path),
        "alpha": float(best_row["alpha"]),
    }
    residual_artifacts.append({
        "region": region,
        "model_name": model_name,
        "alpha": float(best_row["alpha"]),
        "validation_mae": float(best_row["MAE"]),
        "validation_rmse": float(best_row["RMSE"]),
        "saved_model": str(model_path),
        "saved_metadata": str(metadata_path),
    })

residual_artifacts_df = pd.DataFrame(residual_artifacts).sort_values("region").reset_index(drop=True)
display(residual_artifacts_df)
residual_final_models

## 7. Multi-Horizon Models: 5m, 15m, 30m

This section extends the notebook to train residual-blend models for multiple horizons:
- `t+1` = 5 minutes
- `t+3` = 15 minutes
- `t+6` = 30 minutes

The idea is to compare accuracy and stability across short-horizon forecasts using the same feature family and evaluation flow.

In [ ]:
HORIZONS = {
    1: "5m",
    3: "15m",
    6: "30m",
}


def build_multi_horizon_frames(train_df: pd.DataFrame, val_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    combined = pd.concat(
        [
            train_df.assign(dataset_split="train"),
            val_df.assign(dataset_split="val"),
        ],
        ignore_index=True,
    ).sort_values("timestamp").reset_index(drop=True)

    for region in regions:
        current_col = f"{region}_price_dollar_per_mwh"
        for horizon in HORIZONS:
            combined[f"{region}_target_price_t_plus_{horizon}"] = combined[current_col].shift(-horizon)

    enhanced = add_enhanced_short_horizon_features(combined)
    train_multi = enhanced[enhanced["dataset_split"] == "train"].copy().reset_index(drop=True)
    val_multi = enhanced[enhanced["dataset_split"] == "val"].copy().reset_index(drop=True)

    return combined, train_multi, val_multi


combined_multi_df, train_multi_df, val_multi_df = build_multi_horizon_frames(train_df, val_df)
combined_multi_df.shape, train_multi_df.shape, val_multi_df.shape

In [ ]:
multi_horizon_target_counts = []

for region in regions:
    for horizon, horizon_label in HORIZONS.items():
        target_col = f"{region}_target_price_t_plus_{horizon}"
        multi_horizon_target_counts.append({
            "region": region,
            "horizon_steps": horizon,
            "horizon_label": horizon_label,
            "train_count": int(train_multi_df[target_col].notna().sum()),
            "val_count": int(val_multi_df[target_col].notna().sum()),
        })

multi_horizon_target_counts_df = pd.DataFrame(multi_horizon_target_counts).sort_values(["region", "horizon_steps"])
display(multi_horizon_target_counts_df)

## 8. Train Residual-Blend Models Across All Horizons

For each region and horizon:
- baseline = current price
- target = future price at the chosen horizon
- residual model = future price minus current price
- final prediction = current price + `alpha * predicted_residual`

`alpha` is tuned on the validation set separately for each horizon.

In [ ]:
multi_horizon_alpha_grid = np.linspace(0.0, 1.0, 21)
multi_horizon_results = []
multi_horizon_predictions = {}
multi_horizon_models = {}
multi_horizon_feature_map = {}

for horizon, horizon_label in HORIZONS.items():
    for region in regions:
        target = f"{region}_target_price_t_plus_{horizon}"
        current = f"{region}_price_dollar_per_mwh"
        feature_cols = get_residual_feature_cols(region, train_multi_df)
        multi_horizon_feature_map[(region, horizon)] = feature_cols

        train_subset = train_multi_df[train_multi_df[target].notna()].copy()
        val_subset = val_multi_df[val_multi_df[target].notna()].copy()

        X_train = train_subset[feature_cols]
        X_val = val_subset[feature_cols]
        y_train_target = train_subset[target]
        y_val_target = val_subset[target]
        baseline_pred = val_subset[current].to_numpy()
        y_train_residual = y_train_target - train_subset[current]

        baseline_metrics = evaluate_predictions(y_val_target, baseline_pred)

        model = LGBMRegressor(
            objective="regression_l1",
            n_estimators=600,
            learning_rate=0.02,
            num_leaves=7,
            min_child_samples=200,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            force_col_wise=True,
        )
        model.fit(X_train, y_train_residual)

        predicted_residual = model.predict(X_val)
        multi_horizon_predictions[(region, horizon, "raw_residual")] = predicted_residual
        multi_horizon_models[(region, horizon)] = model

        best_choice = None
        best_prediction = None
        for alpha in multi_horizon_alpha_grid:
            blended_pred = baseline_pred + alpha * predicted_residual
            metrics = evaluate_predictions(y_val_target, blended_pred)
            candidate = {
                "region": region,
                "horizon_steps": horizon,
                "horizon_label": horizon_label,
                "model": "lightgbm_residual_blend",
                "alpha": float(alpha),
                "feature_count": len(feature_cols),
                "train_count": len(train_subset),
                "val_count": len(val_subset),
                "baseline_MAE": baseline_metrics["MAE"],
                "baseline_RMSE": baseline_metrics["RMSE"],
                **metrics,
            }
            candidate["MAE_improvement_vs_baseline"] = candidate["baseline_MAE"] - candidate["MAE"]
            candidate["RMSE_improvement_vs_baseline"] = candidate["baseline_RMSE"] - candidate["RMSE"]

            if best_choice is None or candidate["RMSE"] < best_choice["RMSE"] or (
                np.isclose(candidate["RMSE"], best_choice["RMSE"]) and candidate["MAE"] < best_choice["MAE"]
            ):
                best_choice = candidate
                best_prediction = blended_pred

        multi_horizon_results.append(best_choice)
        multi_horizon_predictions[(region, horizon, "best_blend")] = best_prediction

multi_horizon_results_df = pd.DataFrame(multi_horizon_results).sort_values(["horizon_steps", "region"]).reset_index(drop=True)
display(multi_horizon_results_df)

In [ ]:
horizon_summary_df = (
    multi_horizon_results_df[[
        "region",
        "horizon_steps",
        "horizon_label",
        "alpha",
        "MAE",
        "RMSE",
        "baseline_MAE",
        "baseline_RMSE",
        "MAE_improvement_vs_baseline",
        "RMSE_improvement_vs_baseline",
    ]]
    .sort_values(["region", "horizon_steps"])
    .reset_index(drop=True)
)

display(horizon_summary_df)

best_horizon_per_region_df = (
    multi_horizon_results_df.sort_values(["region", "RMSE", "MAE"])
    .groupby("region", as_index=False)
    .first()
    .sort_values("region")
)

print("Best horizon per region by validation RMSE")
display(best_horizon_per_region_df[[
    "region",
    "horizon_steps",
    "horizon_label",
    "alpha",
    "MAE",
    "RMSE",
    "MAE_improvement_vs_baseline",
    "RMSE_improvement_vs_baseline",
]])

In [ ]:
fig, axes = plt.subplots(len(regions), 1, figsize=(16, 3 * len(regions)), sharex=False)
for ax, region in zip(axes, regions):
    subset = multi_horizon_results_df[multi_horizon_results_df["region"] == region].sort_values("horizon_steps")
    sns.barplot(data=subset, x="horizon_label", y="RMSE", ax=ax, palette="mako")
    ax.plot(subset["horizon_label"], subset["baseline_RMSE"], color="crimson", marker="o", label="baseline RMSE")
    ax.set_title(f"{region}: RMSE by forecast horizon")
    ax.set_xlabel("Forecast horizon")
    ax.set_ylabel("Validation RMSE")
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
for region in regions:
    fig, axes = plt.subplots(len(HORIZONS), 1, figsize=(16, 3 * len(HORIZONS)), sharex=False)
    if len(HORIZONS) == 1:
        axes = [axes]

    for ax, (horizon, horizon_label) in zip(axes, HORIZONS.items()):
        target = f"{region}_target_price_t_plus_{horizon}"
        val_subset = val_multi_df[val_multi_df[target].notna()].copy()
        sample = pd.DataFrame({
            "actual": val_subset[target].head(250).values,
            "baseline": val_subset[f"{region}_price_dollar_per_mwh"].head(250).values,
            "prediction": np.asarray(multi_horizon_predictions[(region, horizon, "best_blend")])[:250],
        })
        alpha = multi_horizon_results_df.loc[
            (multi_horizon_results_df["region"] == region) & (multi_horizon_results_df["horizon_steps"] == horizon),
            "alpha",
        ].iloc[0]

        ax.plot(sample.index, sample["actual"], label="actual", linewidth=1.8)
        ax.plot(sample.index, sample["baseline"], label="baseline current price", linewidth=1.2)
        ax.plot(sample.index, sample["prediction"], label=f"residual blend alpha={alpha:.2f}", linewidth=1.4)
        ax.set_title(f"{region} diagnostics at {horizon_label}")
        ax.legend(loc="upper right")

    plt.tight_layout()
    plt.show()

## 9. Save Multi-Horizon Models

This section saves model artifacts for every region and every horizon, so Phase 3 can later choose whether to serve:
- only 5-minute forecasts
- only 30-minute forecasts
- or all horizons together.

In [ ]:
MULTI_HORIZON_MODEL_DIR = MODEL_DIR / "multi_horizon"
MULTI_HORIZON_MODEL_DIR.mkdir(exist_ok=True)

multi_horizon_artifacts = []
multi_horizon_final_models = {}

for horizon, horizon_label in HORIZONS.items():
    for region in regions:
        model = multi_horizon_models[(region, horizon)]
        best_row = multi_horizon_results_df[
            (multi_horizon_results_df["region"] == region) &
            (multi_horizon_results_df["horizon_steps"] == horizon)
        ].iloc[0]

        model_name = "lightgbm_residual_blend"
        stem = f"{region.lower()}_{model_name}_tplus{horizon}"
        model_path = MULTI_HORIZON_MODEL_DIR / f"{stem}.pkl"
        metadata_path = MULTI_HORIZON_MODEL_DIR / f"{stem}_metadata.json"

        joblib.dump(model, model_path)
        metadata = {
            "region": region,
            "model_name": model_name,
            "target_column": f"{region}_target_price_t_plus_{horizon}",
            "current_price_column": f"{region}_price_dollar_per_mwh",
            "horizon_steps": int(horizon),
            "horizon_minutes": int(horizon * 5),
            "horizon_label": horizon_label,
            "feature_columns": multi_horizon_feature_map[(region, horizon)],
            "prediction_rule": "current_price + alpha * model_predicted_residual",
            "alpha": float(best_row["alpha"]),
            "validation_mae": float(best_row["MAE"]),
            "validation_rmse": float(best_row["RMSE"]),
            "baseline_mae": float(best_row["baseline_MAE"]),
            "baseline_rmse": float(best_row["baseline_RMSE"]),
        }
        metadata_path.write_text(json.dumps(metadata, indent=2))

        multi_horizon_final_models.setdefault(region, {})[horizon_label] = {
            "type": "trained_model",
            "model_name": model_name,
            "model_path": str(model_path),
            "metadata_path": str(metadata_path),
            "alpha": float(best_row["alpha"]),
        }

        multi_horizon_artifacts.append({
            "region": region,
            "horizon_steps": horizon,
            "horizon_label": horizon_label,
            "alpha": float(best_row["alpha"]),
            "validation_mae": float(best_row["MAE"]),
            "validation_rmse": float(best_row["RMSE"]),
            "baseline_mae": float(best_row["baseline_MAE"]),
            "baseline_rmse": float(best_row["baseline_RMSE"]),
            "saved_model": str(model_path),
            "saved_metadata": str(metadata_path),
        })

multi_horizon_artifacts_df = pd.DataFrame(multi_horizon_artifacts).sort_values(["horizon_steps", "region"]).reset_index(drop=True)
display(multi_horizon_artifacts_df)
multi_horizon_final_models